# Constrained Spherical Deconvolution (CSD)

DTI fails wherever two or more fibre populations cross within one voxel — which happens in roughly **90% of white matter voxels** at typical 2 mm resolution. CSD was developed to solve this.

## The idea

Instead of fitting a single ellipsoid, CSD estimates a **Fibre Orientation Distribution (FOD)**: a function defined on the sphere that describes *how much fibre* points in each direction.

The signal in voxel $v$ is modelled as:

$$S(\hat{g}) = \int_{S^2} f(\hat{n}) \, R(\hat{g} \cdot \hat{n}) \, d\hat{n}$$

where:
- $f(\hat{n})$ is the FOD — what we want to estimate
- $R(\hat{g} \cdot \hat{n})$ is the **response function** — the signal a single perfectly coherent fibre bundle oriented along $\hat{n}$ would produce when measured with gradient $\hat{g}$
- The integral is a **spherical convolution** — hence *deconvolution* to invert it

### The constraint
The raw deconvolution problem is ill-posed. CSD adds the constraint $f(\hat{n}) \geq 0$ everywhere (no negative fibre density), which regularises the solution.

### Multi-shell multi-tissue (MSMT-CSD)
HCP data has three shells. MSMT-CSD uses all shells simultaneously to separate:
- WM FOD (anisotropic)
- GM partial volume (isotropic, medium diffusivity)
- CSF partial volume (isotropic, high diffusivity)

This produces cleaner FODs near tissue boundaries.

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import mrtrix_dwi2response, mrtrix_dwi2fod, run

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
csd_dir  = Path('../../data/hcp/100307/csd')
csd_dir.mkdir(parents=True, exist_ok=True)

# Use eddy-corrected MIF if available
dwi_mif = str(prep_dir / 'dwi_eddy.mif') if (prep_dir / 'dwi_eddy.mif').exists() \
          else str(prep_dir / 'dwi_raw.mif')
mask    = str(data_dir / 'nodif_brain_mask.nii.gz')

# Response function output files
resp_wm  = str(csd_dir / 'response_wm.txt')
resp_gm  = str(csd_dir / 'response_gm.txt')
resp_csf = str(csd_dir / 'response_csf.txt')

# FOD output files
wm_fod   = str(csd_dir / 'wmfod.mif')
gm_fod   = str(csd_dir / 'gmfod.mif')
csf_fod  = str(csd_dir / 'csffod.mif')

print('Setup complete.')

## Step 1: Estimate the response function

The response function describes the signal of a **single coherent fibre population**. It is estimated from the data itself using the `dhollander` algorithm, which identifies voxels that are very likely single-fibre WM, pure GM, or pure CSF.

In [ ]:
# ─── [MRtrix3] dwi2response (dhollander algorithm) ───────────────────────────
#
# This is the recommended algorithm for MSMT pipelines.
# It does NOT require a T1w image — it estimates tissue classes
# directly from the DWI signal.
#
# Output: three .txt files with the spherical harmonic coefficients
# of the response function for WM, GM, and CSF.

print('Estimating response functions (dhollander) ...')
mrt_resp_cmd = [
    'dwi2response', 'dhollander',
    dwi_mif,
    resp_wm, resp_gm, resp_csf,
    '-voxels', str(csd_dir / 'response_voxels.mif'),  # QC: which voxels were used
    '-force',
]
print(' '.join(mrt_resp_cmd))

result = subprocess.run(mrt_resp_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✓ Response functions estimated')

In [ ]:
# Inspect the response functions
for label, path in [('WM', resp_wm), ('GM', resp_gm), ('CSF', resp_csf)]:
    if Path(path).exists():
        print(f'=== {label} response function ===')
        print(open(path).read())
        print()

In [ ]:
# ─── [DIPY] Auto-response estimation ─────────────────────────────────────────
#
# DIPY has several response function estimators.
# 'auto_response_ssst' is for single-shell; for MSMT use 'recursive_response'.

from dipy.reconst.csdeconv import auto_response_ssst, recursive_response
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table

bvals, bvecs = read_bvals_bvecs(
    str(data_dir / 'bvals'),
    str(data_dir / 'bvecs')
)
gtab = gradient_table(bvals, bvecs)

img   = nib.load(str(data_dir / 'data.nii.gz'))
data  = img.get_fdata()
maskd = nib.load(mask).get_fdata().astype(bool)

# Single-shell: use only b=1000
sel_1000 = (bvals < 50) | ((bvals > 900) & (bvals < 1100))
gtab_1000 = gradient_table(bvals[sel_1000], bvecs[sel_1000])
data_1000 = data[..., sel_1000]

response_dipy, ratio = auto_response_ssst(
    gtab_1000, data_1000, roi_radii=10, fa_thr=0.7
)

print('[DIPY] Single-shell response function:')
print(f'  Eigenvalues : {response_dipy[0]}')
print(f'  S0          : {response_dipy[1]:.1f}')
print(f'  FA ratio    : {ratio:.3f}  (should be ≈ 0.7–0.8 for good WM voxels)')

## Step 2: Estimate FODs

### MRtrix3: MSMT-CSD (multi-shell multi-tissue)

In [ ]:
# ─── [MRtrix3] dwi2fod (msmt_csd) ────────────────────────────────────────────
#
# msmt_csd uses all available shells simultaneously.
# This gives better WM/GM/CSF separation than single-shell CSD.

print('Computing MSMT-CSD FODs ...')
mrt_fod_cmd = [
    'dwi2fod', 'msmt_csd',
    dwi_mif,
    resp_wm,  wm_fod,
    resp_gm,  gm_fod,
    resp_csf, csf_fod,
    '-mask', mask,
    '-force',
]
print(' '.join(mrt_fod_cmd))

result = subprocess.run(mrt_fod_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✓ FODs computed')

### DIPY: Single-shell CSD

In [ ]:
# ─── [DIPY] ConstrainedSphericalDeconvModel ───────────────────────────────────
from dipy.reconst.csdeconv import ConstrainedSphericalDeconvModel
from dipy.data import get_sphere

sphere = get_sphere('symmetric724')

print('Fitting CSD model (b=1000 shell) ...')
csd_model = ConstrainedSphericalDeconvModel(gtab_1000, response_dipy, sh_order=8)
csd_fit   = csd_model.fit(data_1000, mask=maskd)

# FOD peaks: dominant fibre directions in each voxel
from dipy.direction import peaks_from_model

csd_peaks = peaks_from_model(
    model=csd_model,
    data=data_1000,
    sphere=sphere,
    relative_peak_threshold=0.5,
    min_separation_angle=25,
    mask=maskd,
    return_sh=True,
    return_odf=False,
    npeaks=3,
    normalize_peaks=True,
)

print('✓ DIPY CSD complete')
print(f'  GFA mean (brain): {csd_peaks.gfa[maskd].mean():.3f}')
print(f'  GFA is the CSD equivalent of FA — generalised fractional anisotropy')

# Save GFA
nib.save(nib.Nifti1Image(csd_peaks.gfa.astype(np.float32), img.affine),
         str(csd_dir / 'dipy_GFA.nii.gz'))

## Visualise the FODs

In [ ]:
# ─── [DIPY] FOD peaks in 2D (matplotlib) ─────────────────────────────────────
#
# Show a small region of the corpus callosum / corona radiata
# where crossing fibres are common.

from dipy.reconst.shm import sh_to_sharp

# Select a coronal slice through the corpus callosum
cc_y = data.shape[1] // 2   # mid-sagittal-ish
z_slice = data.shape[2] // 2

# Count peaks per voxel in a coronal slice
peak_dirs = csd_peaks.peak_dirs             # shape: (x, y, z, 3peaks, 3xyz)
peak_vals = csd_peaks.peak_values           # shape: (x, y, z, 3peaks)

# Number of valid peaks (value > 0)
n_peaks = np.sum(peak_vals > 0, axis=-1)    # shape: (x, y, z)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(n_peaks[:, cc_y, :].T, cmap='hot', origin='lower', vmin=0, vmax=3)
axes[0].set_title('DIPY CSD: # fibre peaks per voxel\n(coronal slice)', fontsize=11)
axes[0].axis('off')
plt.colorbar(axes[0].images[0], ax=axes[0], label='# peaks')

gfa_vol = nib.load(str(csd_dir / 'dipy_GFA.nii.gz')).get_fdata()
axes[1].imshow(gfa_vol[:, cc_y, :].T, cmap='hot', origin='lower', vmin=0, vmax=1)
axes[1].set_title('DIPY CSD: GFA\n(coronal slice)', fontsize=11)
axes[1].axis('off')
plt.colorbar(axes[1].images[0], ax=axes[1], label='GFA')

plt.suptitle('CSD reveals multi-fibre architecture hidden by DTI', fontsize=12)
plt.tight_layout()
plt.show()

crossing_frac = (n_peaks[maskd] > 1).mean() * 100
print(f'{crossing_frac:.0f}% of brain voxels contain crossing fibres (>1 peak)')

## Normalise the FODs (MRtrix3 step)

Before tractography, MRtrix3 recommends **normalising the intensity of the FODs** across subjects using `mtnormalise`. This corrects for global signal differences due to scanner drift or different participant coil loading.

In [ ]:
# ─── [MRtrix3] mtnormalise ────────────────────────────────────────────────────
wm_fod_norm  = str(csd_dir / 'wmfod_norm.mif')
gm_fod_norm  = str(csd_dir / 'gmfod_norm.mif')
csf_fod_norm = str(csd_dir / 'csffod_norm.mif')

norm_cmd = [
    'mtnormalise',
    wm_fod,  wm_fod_norm,
    gm_fod,  gm_fod_norm,
    csf_fod, csf_fod_norm,
    '-mask', mask,
    '-force',
]
print(' '.join(norm_cmd))

result = subprocess.run(norm_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✓ FODs normalised. Use the *_norm.mif files for tractography.')

---

## Summary

| Aspect | MRtrix3 msmt_csd | DIPY ConstrainedSphericalDeconvModel |
|---|---|---|
| Multi-shell support | Yes (native) | Requires manual shell selection |
| Tissue separation | WM + GM + CSF | WM only |
| Speed | Fast (C++) | Moderate (Python) |
| Output | .mif (SH coefficients) | NumPy arrays |
| Normalisation | mtnormalise | Manual |
| **Recommendation** | **Use for tractography** | **Use for QC, research, learning** |

**FSL has no CSD implementation** — this is a clear strength of MRtrix3.

**Next**: [DTI vs CSD: when to use which →](03_comparison.ipynb)